# Lista 5 — Zadanie 3: Eksploracja modeli encoder-only (20 pkt)

Porównujemy co najmniej **dwa aspekty**:
1. **Różne modele** klasyfikujące (HerBERT vs inny model z Hugging Face)
2. **Parametr `max_length`** — jak długi kontekst wpływa na wyniki (dobór na podstawie analizy długości tekstów)
3. **Temperatura** (`temperature`) — skalowanie logitów przed softmax (domyślnie 1.0)

Wyniki zestawiamy w tabeli porównawczej i analizujemy trudne przypadki.

In [ ]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas matplotlib

In [ ]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions, print_evaluation

## Krok 1: Przygotowanie danych

In [ ]:
examples = load_polemo_test()
sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]

device = 0 if torch.cuda.is_available() else -1
print(f"Próbek: {len(sentences)} | Urządzenie: {'GPU' if device == 0 else 'CPU'}")

## Analiza długości tekstów

Przed doborem `max_length` sprawdzamy rozkład długości recenzji (w słowach), żeby opierać eksperyment na statystykach zbioru.

In [ ]:
# Analiza długości tekstów (w słowach)
lengths = [len(s.split()) for s in sentences]

print(f"Najkrótszy tekst (słowa): {min(lengths)}")
print(f"Najdłuższy tekst (słowa): {max(lengths)}")
print(f"Średnia długość: {sum(lengths) / len(lengths):.1f}")
print(f"Mediana: {pd.Series(lengths).median():.0f}")
print(f"90. percentyl: {pd.Series(lengths).quantile(0.9):.0f}")

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Liczba słów")
plt.ylabel("Liczba recenzji")
plt.title("Rozkład długości tekstów w zbiorze testowym")
plt.axvline(128, color="red", linestyle="--", label="max_length=128 (orientacyjnie)")
plt.axvline(512, color="green", linestyle="--", label="max_length=512")
plt.legend()
plt.tight_layout()
plt.show()

## Krok 2: Funkcja pomocnicza do eksperymentów

In [ ]:
def run_encoder_experiment(model_name, max_length=512, batch_size=16, temperature=1.0):
    """Ładuje model, klasyfikuje dane i zwraca metryki oraz predykcje."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    device_str = "cuda" if device == 0 else "cpu"
    model.to(device_str)
    model.eval()

    y_pred = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        ).to(device_str)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits / temperature
            probs = F.softmax(logits, dim=-1)
            predictions = torch.argmax(probs, dim=-1)

        for pred_idx in predictions.cpu().tolist():
            label = model.config.id2label[pred_idx]
            mapped = map_text_to_class(label)
            y_pred.append(mapped if mapped else "neutral")

    metrics = evaluate_predictions(y_true, y_pred)
    return metrics, y_pred

## Eksperyment A: Porównanie modeli

| Model | Opis |
|-------|------|
| `Voicelab/herbert-base-cased-sentiment` | HerBERT fine-tuned na sentyment ogólny (recenzje) |
| `bardsai/finance-sentiment-pl-base` | HerBERT fine-tuned na sentyment finansowy (inna domena) |

In [ ]:
MODELS_TO_COMPARE = [
    "Voicelab/herbert-base-cased-sentiment",
    "bardsai/finance-sentiment-pl-base",
]

model_results = []
for model_name in MODELS_TO_COMPARE:
    print(f"\n>>> Uruchamiam: {model_name}")
    try:
        metrics, _ = run_encoder_experiment(model_name, max_length=512)
        print_evaluation(metrics, title=model_name)
        model_results.append({
            "eksperyment": "model",
            "wariant": model_name.split("/")[-1],
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        })
    except Exception as e:
        print(f"Błąd dla {model_name}: {e}")
        print("Sprawdź nazwę modelu na https://huggingface.co/models?pipeline_tag=text-classification&language=pl")

## Eksperyment B: Wpływ max_length

Sprawdzamy, czy skracanie kontekstu (128 vs 512 tokenów) zmienia jakość klasyfikacji.

In [ ]:
BASE_MODEL = "Voicelab/herbert-base-cased-sentiment"
MAX_LENGTHS = [128, 256, 512]

length_results = []
y_pred_baseline = None
for max_len in MAX_LENGTHS:
    print(f"\n>>> max_length = {max_len}")
    metrics, y_pred = run_encoder_experiment(BASE_MODEL, max_length=max_len)
    if max_len == 512:
        y_pred_baseline = y_pred
    print_evaluation(metrics, title=f"max_length={max_len}")
    length_results.append({
        "eksperyment": "max_length",
        "wariant": str(max_len),
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
    })

## Analiza trudnych przypadków

Sprawdzamy, gdzie model (HerBERT, `max_length=512`) się myli — to pomaga zrozumieć ograniczenia klasyfikatora.

In [ ]:
df_results = pd.DataFrame({"text": sentences, "true": y_true, "pred": y_pred_baseline})
errors = df_results[df_results["true"] != df_results["pred"]]

print(f"Liczba błędów: {len(errors)} / {len(df_results)} ({100 * len(errors) / len(df_results):.1f}%)")
print(f"\nRozkład błędów (true → pred):")
print(errors.groupby(["true", "pred"]).size().sort_values(ascending=False).head(10))

print("\nPrzykłady błędów:")
for _, row in errors.head(5).iterrows():
    print(f"\n  Prawda: {row['true']} | Predykcja: {row['pred']}")
    print(f"  Tekst: {row['text'][:200]}{'...' if len(row['text']) > 200 else ''}")

## Krok 3: Tabela porównawcza

In [ ]:
comparison_df = pd.DataFrame(model_results + length_results)
comparison_df

## Podsumowanie (do raportu)

- **Analiza długości** — rozkład słów w recenzjach uzasadnia testowanie `max_length` (np. 128 vs 512), zamiast arbitralnego doboru.
- **Model z tej samej domeny** (recenzje) zwykle radzi sobie lepiej niż model z innej domeny (np. finanse).
- **max_length** — zbyt krótki kontekst może obcinać istotne fragmenty długich recenzji.
- **Temperatura** — parametr skaluje logity przed softmax; przy `temperature=1.0` wynik jest równoważny standardowej klasyfikacji argmax.
- **Trudne przypadki** — błędy często dotyczą recenzji neutralnych lub mieszanych (np. pozytywny opis z negatywnym wnioskiem).
- Klasa **neutral** jest najtrudniejsza dla obu modeli (niski recall).
- Inne modele PL: [Hugging Face — text-classification (pl)](https://huggingface.co/models?pipeline_tag=text-classification&language=pl).